In [1]:
# Agentic Weekly Learning Insights

In this notebook, I build a simple agentic workflow that reads weekly study behavior data and generates personalized learning insights.

The goal is to convert numerical study patterns into plain-language recommendations.

Instead of only showing charts or model outputs, this workflow summarizes the learner's weekly behavior and suggests one specific improvement for the next week.

SyntaxError: unterminated string literal (detected at line 7) (3383444273.py, line 7)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv("../data/processed/cleaned_study_sessions.csv")

df.head()

,session_id,date,day_of_week,start_time,end_time,hour_of_day,time_of_day,duration_min,topic,topic_type,difficulty_1_5,sleep_hours,energy_before_1_10,distraction_level_1_10,break_before_days,location,device_used,completion_percent,focus_score_1_10,notes,year,month,week,weekday_number
0,S0001,2024-01-08,Monday,15:10,16:10,15,Afternoon,60,Machine Learning,ML,4,7.8,6,2,0,Study Room,Laptop + Notebook,77,6,"Decent session, some distractions but useful p...",2024,1,2,0
1,S0002,2024-01-09,Tuesday,16:00,16:50,16,Afternoon,50,Python Programming,Programming,3,7.5,6,5,0,Study Room,Laptop,68,5,"Low energy, needed more breaks than expected",2024,1,2,1
2,S0003,2024-01-10,Wednesday,18:15,19:15,18,Evening,60,German Language,Language,3,7.3,5,2,0,Cafe,Laptop + Notebook,90,6,"Decent session, some distractions but useful p...",2024,1,2,2
3,S0004,2024-01-11,Thursday,15:45,16:30,15,Afternoon,45,Linear Algebra,Math,4,7.2,7,4,0,Home Desk,Laptop + Notebook,62,5,"Low energy, needed more breaks than expected",2024,1,2,3
4,S0005,2024-01-12,Friday,10:00,11:00,10,Morning,60,SQL Practice,Programming,3,8.5,7,3,0,University Library,Laptop + Notebook,91,8,"High clarity, strong concentration, completed ...",2024,1,2,4


In [ ]:
df["date"] = pd.to_datetime(df["date"])

df["date"].min(), df["date"].max()

(Timestamp('2024-01-08 00:00:00'), Timestamp('2025-12-28 00:00:00'))

In [ ]:
weekly_summary = (
    df.set_index("date")
    .resample("W")
    .agg(
        sessions_count=("session_id", "count"),
        avg_focus_score=("focus_score_1_10", "mean"),
        avg_duration_min=("duration_min", "mean"),
        avg_sleep_hours=("sleep_hours", "mean"),
        avg_energy_before=("energy_before_1_10", "mean"),
        avg_distraction_level=("distraction_level_1_10", "mean"),
        avg_completion_percent=("completion_percent", "mean"),
        total_study_minutes=("duration_min", "sum"),
        avg_break_before_days=("break_before_days", "mean")
    )
    .reset_index()
)

weekly_summary.head()

,date,sessions_count,avg_focus_score,avg_duration_min,avg_sleep_hours,avg_energy_before,avg_distraction_level,avg_completion_percent,total_study_minutes,avg_break_before_days
0,2024-01-14,8,5.500000,70.000000,7.637500,6.125000,3.375000,79.000000,560,0.000000
1,2024-01-21,8,5.750000,56.250000,6.675000,5.000000,3.125000,85.250000,450,0.250000
2,2024-01-28,3,5.666667,66.666667,6.633333,6.666667,2.666667,73.333333,200,1.333333
3,2024-02-04,3,4.333333,66.666667,7.200000,5.333333,4.333333,71.000000,200,1.333333
4,2024-02-11,6,6.333333,100.000000,7.083333,6.500000,3.666667,90.000000,600,0.166667


In [ ]:
print("Weekly summary shape:", weekly_summary.shape)

weekly_summary.tail(10)

Weekly summary shape: (103, 10)


,date,sessions_count,avg_focus_score,avg_duration_min,avg_sleep_hours,avg_energy_before,avg_distraction_level,avg_completion_percent,total_study_minutes,avg_break_before_days
93,2025-10-26,6,6.500000,83.333333,6.866667,6.166667,3.000000,81.833333,500,0.333333
94,2025-11-02,9,6.000000,78.888889,6.722222,6.555556,3.777778,85.111111,710,0.000000
95,2025-11-09,4,5.000000,82.500000,7.675000,5.000000,4.250000,80.750000,330,1.000000
96,2025-11-16,3,7.333333,75.000000,6.666667,7.333333,2.666667,82.000000,225,1.333333
97,2025-11-23,6,4.833333,70.833333,7.450000,5.833333,4.166667,70.500000,425,0.666667
98,2025-11-30,9,6.222222,71.111111,7.055556,5.666667,3.000000,88.666667,640,0.111111
99,2025-12-07,9,6.222222,80.000000,7.266667,6.555556,2.666667,85.111111,720,0.222222
100,2025-12-14,5,6.400000,93.000000,7.620000,6.400000,3.800000,86.200000,465,0.400000
101,2025-12-21,5,6.400000,93.000000,6.480000,6.400000,3.200000,89.000000,465,0.600000
102,2025-12-28,6,6.000000,62.500000,7.050000,5.666667,3.333333,80.166667,375,0.333333


In [ ]:
selected_week = weekly_summary.iloc[-1]

selected_week

date                      2025-12-28 00:00:00
sessions_count                              6
avg_focus_score                           6.0
avg_duration_min                         62.5
avg_sleep_hours                          7.05
avg_energy_before                    5.666667
avg_distraction_level                3.333333
avg_completion_percent              80.166667
total_study_minutes                       375
avg_break_before_days                0.333333
Name: 102, dtype: object

In [ ]:
week_text = f"""
Weekly Learning Summary

Week ending: {selected_week['date'].date()}

Number of study sessions: {selected_week['sessions_count']}
Average focus score: {selected_week['avg_focus_score']:.2f}/10
Average session duration: {selected_week['avg_duration_min']:.1f} minutes
Average sleep hours: {selected_week['avg_sleep_hours']:.1f}
Average energy before studying: {selected_week['avg_energy_before']:.2f}/10
Average distraction level: {selected_week['avg_distraction_level']:.2f}/10
Average completion percentage: {selected_week['avg_completion_percent']:.1f}%
Total study time: {selected_week['total_study_minutes']:.0f} minutes
Average break before sessions: {selected_week['avg_break_before_days']:.2f} days
"""

print(week_text)


Weekly Learning Summary

Week ending: 2025-12-28

Number of study sessions: 6
Average focus score: 6.00/10
Average session duration: 62.5 minutes
Average sleep hours: 7.0
Average energy before studying: 5.67/10
Average distraction level: 3.33/10
Average completion percentage: 80.2%
Total study time: 375 minutes
Average break before sessions: 0.33 days



In [ ]:
def generate_basic_weekly_insight(week):
    insights = []

    if week["avg_focus_score"] >= 7:
        insights.append("Focus was strong this week. The learner maintained good concentration during study sessions.")
    elif week["avg_focus_score"] >= 5:
        insights.append("Focus was moderate this week. There is room to improve consistency and session quality.")
    else:
        insights.append("Focus was low this week. The learner may need to adjust study timing, reduce distractions, or improve rest.")

    if week["avg_distraction_level"] > 6:
        insights.append("Distraction level was high, which may have reduced focus and completion percentage.")
    
    if week["avg_sleep_hours"] < 6.5:
        insights.append("Average sleep was below the recommended level, which may have affected attention and energy.")

    if week["avg_completion_percent"] < 70:
        insights.append("Completion percentage was relatively low, meaning planned study sessions were not fully completed.")

    if week["sessions_count"] < 3:
        insights.append("The number of study sessions was low, which may indicate a consistency gap.")

    recommendation = "Recommended action: Schedule at least one focused morning study session next week and reduce distractions before starting."

    return "\n".join(insights) + "\n\n" + recommendation


basic_insight = generate_basic_weekly_insight(selected_week)

print(basic_insight)

Focus was moderate this week. There is room to improve consistency and session quality.

Recommended action: Schedule at least one focused morning study session next week and reduce distractions before starting.


In [ ]:
import os

os.makedirs("../outputs/weekly_insights", exist_ok=True)

with open("../outputs/weekly_insights/basic_weekly_insight.txt", "w") as file:
    file.write(week_text)
    file.write("\n")
    file.write(basic_insight)

print("Basic weekly insight saved successfully.")

Basic weekly insight saved successfully.
